# Runnable Passthrough Reference

Developer-facing statements defined in `langchain_core.runnables.passthrough`.

# `identity`

Returns the supplied value unchanged.

```python
identity(
    x: Other, # Input value returned without modification
) -> Other # Return the original input value
```

---

# `aidentity`

Asynchronously returns the supplied value unchanged.

```python
async aidentity(
    x: Other, # Input value returned without modification
) -> Other # Return the original input value
```

---



# `RunnablePassthrough: RunnableSerializable[Other, Other]`

`RunnablePassthrough` passes its input through unchanged.

It may also execute a synchronous or asynchronous side-effect function without changing the returned value.

For dictionary inputs, `assign()` can add new keys while preserving the original dictionary.

## Type Parameter

```python
Other # Input and output type passed through unchanged
```

## Fields

```python
input_type: type[Other] | None = None # Optional explicit input and output type
func: Callable[[Other], None] | Callable[[Other, RunnableConfig], None] | None = None # Optional synchronous side-effect function
afunc: Callable[[Other], Awaitable[None]] | Callable[[Other, RunnableConfig], Awaitable[None]] | None = None # Optional asynchronous side-effect function
```

## Constructor

```python
RunnablePassthrough(
    func: Callable[[Other], None]
    | Callable[[Other, RunnableConfig], None]
    | Callable[[Other], Awaitable[None]]
    | Callable[[Other, RunnableConfig], Awaitable[None]]
    | None = None, # Optional synchronous or asynchronous side-effect function
    afunc: Callable[[Other], Awaitable[None]]
    | Callable[[Other, RunnableConfig], Awaitable[None]]
    | None = None, # Optional native asynchronous side-effect function
    *,
    input_type: type[Other] | None = None, # Optional explicit input and output type
    **kwargs: Any, # Additional serializable model fields
) -> None # Initialize the passthrough Runnable
```

When `func` is asynchronous, it is stored as `afunc`.

## Overridden Properties and Methods

### `is_lc_serializable`

Returns `True`, indicating that the class supports LangChain serialization.

### `get_lc_namespace`

Returns the LangChain Runnable serialization namespace.

### `InputType`

Returns `input_type` when supplied; otherwise, returns `Any`.

### `OutputType`

Returns `input_type` when supplied; otherwise, returns `Any`.

### `assign`

Creates a `RunnableAssign` that merges a dictionary input with new values produced by named Runnables, callables, or mappings.

### `invoke`

Synchronously executes `func` when supplied and returns the original input unchanged.

### `ainvoke`

Executes `afunc` when supplied, otherwise executes `func`, and returns the original input unchanged.

### `transform`

Yields synchronous input chunks unchanged.

When `func` is supplied, it is called once with the combined final input after all chunks have passed through.

### `atransform`

Asynchronously yields input chunks unchanged.

When `afunc` or `func` is supplied, it is called once with the combined final input after all chunks have passed through.

### `stream`

Synchronously streams the supplied input unchanged.

### `astream`

Asynchronously streams the supplied input unchanged.

## Behaviour

- The returned output is always the original input.
- Side-effect functions do not replace the output.
- Runtime configuration is passed to a side-effect function when its signature accepts it.
- Streaming emits chunks before the side-effect function is called.
- `assign()` requires dictionary input.


In [ ]:
from typing import Any # Import Any for dictionary value types

from langchain_core.runnables import RunnablePassthrough # Import RunnablePassthrough


def log_input(data: dict[str, Any]) -> None: # Define a side-effect function
    print("Received input:", data) # Display the input without modifying it
    return None # Return no replacement value


passthrough: RunnablePassthrough = RunnablePassthrough( # Create the passthrough Runnable
    func=log_input, # Execute the side-effect function during invocation
    input_type=dict, # Declare the expected input type
) # Finish creating RunnablePassthrough

student: dict[str, Any] = { # Create the input dictionary
    "name": "Saad", # Store the student name
    "physics": 80, # Store the physics marks
    "chemistry": 70, # Store the chemistry marks
} # Finish creating the input dictionary

unchanged_result: dict[str, Any] = passthrough.invoke(student) # Run the side effect and return the original input

assigned_runnable = RunnablePassthrough.assign( # Create a Runnable that preserves input and adds fields
    total=lambda data: data["physics"] + data["chemistry"], # Add the total marks field
    result=lambda data: "Pass" if data["physics"] >= 40 and data["chemistry"] >= 40 else "Fail", # Add the result field
) # Finish creating the assignment Runnable

updated_result: dict[str, Any] = assigned_runnable.invoke(student) # Preserve original fields and add calculated fields

print("Unchanged result:", unchanged_result) # Display the unchanged passthrough output

print("Assigned result:", updated_result) # Display the dictionary containing added fields

# `RunnableAssign: RunnableSerializable[dict[str, Any], dict[str, Any]]`

`RunnableAssign` applies a `RunnableParallel` mapper to a dictionary input and merges the mapper output into the original dictionary.

Mapper values replace original values when both dictionaries contain the same key.

## Field

```python
mapper: RunnableParallel[dict[str, Any]] # Named parallel transformations producing assigned values
```

## Constructor

```python
RunnableAssign(
    mapper: RunnableParallel[dict[str, Any]], # Parallel mapper producing new dictionary fields
    **kwargs: Any, # Additional serializable model fields
) -> None # Initialize the dictionary assignment Runnable
```

## Overridden Properties and Methods

### `is_lc_serializable`

Returns `True`, indicating that the class supports LangChain serialization.

### `get_lc_namespace`

Returns the LangChain Runnable serialization namespace.

### `get_name`

Returns an explicit name when supplied.

Otherwise, it generates a name containing the mapper keys.

### `get_input_schema`

Returns the mapper input schema when it represents a dictionary.

Otherwise, it returns the inherited input schema.

### `get_output_schema`

Combines dictionary fields from the mapper input and output schemas when possible.

Otherwise, it returns the mapper output schema or the inherited output schema.

### `config_specs`

Returns the configurable-field specifications exposed by the mapper.

### `get_graph`

Returns the mapper graph with an additional passthrough path connecting the graph input to its output.

### `invoke`

Synchronously merges the original dictionary with values produced by the mapper.

Raises `ValueError` when the input is not a dictionary.

### `ainvoke`

Asynchronously merges the original dictionary with values produced by the mapper.

Raises `ValueError` when the input is not a dictionary.

### `transform`

Streams original dictionary chunks and mapper-generated chunks.

Original keys that are also produced by the mapper are removed from passthrough chunks so the mapper values can replace them.

### `atransform`

Asynchronously streams original dictionary chunks and mapper-generated chunks.

Original keys that are also produced by the mapper are removed from passthrough chunks.

### `stream`

Synchronously streams the merged dictionary output.

### `astream`

Asynchronously streams the merged dictionary output.

## Behaviour

- Input must be a dictionary.
- The mapper receives the complete input dictionary.
- Original fields are preserved unless the mapper produces the same key.
- Mapper keys take precedence over original keys.
- Synchronous mapping begins in a background executor during streaming.
- Asynchronous mapping begins in a background task during asynchronous streaming.

In [ ]:
from typing import Any # Import Any for dictionary value types

from langchain_core.runnables import RunnableLambda, RunnableParallel # Import Runnable components
from langchain_core.runnables.passthrough import RunnableAssign # Import RunnableAssign

student: dict[str, Any] = { # Create the original input dictionary
    "name": "Saad", # Store the student's name
    "physics": 80, # Store the physics marks
    "chemistry": 70, # Store the chemistry marks
    "result": "Pending", # Store an existing value that will be replaced
} # Finish creating the input dictionary

def calculate_total(data: dict[str, Any]) -> int: # Define a function for calculating total marks
    return data["physics"] + data["chemistry"] # Return the sum of both subjects

def calculate_result(data: dict[str, Any]) -> str: # Define a function for calculating the result
    return "Pass" if data["physics"] >= 40 and data["chemistry"] >= 40 else "Fail" # Return Pass or Fail

mapper: RunnableParallel = RunnableParallel( # Create parallel transformations
    total=RunnableLambda(calculate_total), # Produce a new total field
    result=RunnableLambda(calculate_result), # Replace the existing result field
) # Finish creating the parallel mapper

assign_runnable: RunnableAssign = RunnableAssign( # Create the assignment Runnable
    mapper=mapper, # Supply the mapper that generates new fields
) # Finish creating RunnableAssign

updated_student: dict[str, Any] = assign_runnable.invoke(student) # Merge mapper results into the original dictionary

print(updated_student) # Display the updated dictionary

# `RunnablePick: RunnableSerializable[dict[str, Any], Any]`

`RunnablePick` extracts one key or multiple keys from a dictionary input.

A single key returns its value directly.

Multiple keys return a dictionary containing only the selected keys that exist.

## Field

```python
keys: str | list[str] # Single key or list of keys extracted from the input dictionary
```

## Constructor

```python
RunnablePick(
    keys: str | list[str], # Single key or multiple keys selected from the input
    **kwargs: Any, # Additional serializable model fields
) -> None # Initialize the dictionary key-selection Runnable
```

## Overridden Properties and Methods

### `is_lc_serializable`

Returns `True`, indicating that the class supports LangChain serialization.

### `get_lc_namespace`

Returns the LangChain Runnable serialization namespace.

### `get_name`

Returns an explicit name when supplied.

Otherwise, it generates a name containing the selected keys.

### `invoke`

Synchronously extracts the configured key or keys from the input dictionary.

Raises `ValueError` when the input is not a dictionary.

### `ainvoke`

Asynchronously extracts the configured key or keys from the input dictionary.

Raises `ValueError` when the input is not a dictionary.

### `transform`

Extracts the configured key or keys from each synchronous dictionary chunk.

Chunks producing no selected values are not yielded.

### `atransform`

Asynchronously extracts the configured key or keys from each dictionary chunk.

Chunks producing no selected values are not yielded.

### `stream`

Synchronously streams selected values from one dictionary input.

### `astream`

Asynchronously streams selected values from one dictionary input.

## Return Behaviour

```python
Any # Value returned when keys is a single string
AddableDict # Dictionary returned when keys contains multiple matching keys
None # Returned when a single key is missing or no requested list keys exist
```

In [ ]:
from typing import Any # Import Any for dictionary value types

from langchain_core.runnables.passthrough import RunnablePick # Import RunnablePick

student: dict[str, Any] = { # Create the input dictionary
    "name": "Saad", # Store the student name
    "age": 23, # Store the student age
    "city": "Delhi", # Store the student city
    "course": "Python", # Store the student course
} # Finish creating the dictionary

single_picker: RunnablePick = RunnablePick( # Create a Runnable that selects one key
    keys="name", # Select only the name key
) # Finish creating the single-key picker

single_result: Any = single_picker.invoke(student) # Extract the value of the name key

multiple_picker: RunnablePick = RunnablePick( # Create a Runnable that selects multiple keys
    keys=["name", "city"], # Select the name and city keys
) # Finish creating the multiple-key picker

multiple_result: Any = multiple_picker.invoke(student) # Extract the selected keys as a dictionary

missing_picker: RunnablePick = RunnablePick( # Create a Runnable for a missing key
    keys="salary", # Select a key that does not exist
) # Finish creating the missing-key picker

missing_result: Any = missing_picker.invoke(student) # Return None because salary is missing

print(single_result) # Display the single selected value

print(multiple_result) # Display the selected key-value pairs

print(missing_result) # Display None for the missing key

## Developer-Facing Top-Level Statements
```python
identity # Synchronous identity function
aidentity # Asynchronous identity function
RunnablePassthrough # Runnable that returns input unchanged
RunnableAssign # Runnable that adds or replaces dictionary fields
RunnablePick # Runnable that extracts dictionary fields
```